# DisasterAI — Türkçe Model İnce Ayarı (QLoRA)

**Ne yapıyor?** 2.500 örneklik corpus ile 8B Türkçe modeli LoRA yöntemiyle
afet risk iletişimine uyarlıyor (İP2 — plan: `docs/ip2_ince_ayar_plani.md`).

**Kullanım (Ayşe için):**
1. Üst menü → *Çalışma zamanı → Çalışma zamanı türünü değiştir* → **T4 GPU** seç.
2. Hücreleri **sırayla** çalıştır (▶). 2. hücre senden `corpora_egitim.zip` dosyasını isteyecek.
3. Eğitim ~2-4 saat sürer; sekme açık kalsın. Bittiğinde adaptör Drive'ına kaydedilir.
4. Bir hücre hata verirse çıktısını kopyalayıp Claude'a yapıştır.


In [ ]:
#@title 1) GPU kontrolü ve kurulum (≈3 dk)
!nvidia-smi -L
%pip install -q -U transformers peft trl bitsandbytes accelerate datasets
print("kurulum tamam")

In [ ]:
#@title 2) Veriyi yükle — corpora_egitim.zip seçilecek
from google.colab import files
import zipfile, io, json

yuklenen = files.upload()   # corpora_egitim.zip dosyasını seç
ad = next(iter(yuklenen))
with zipfile.ZipFile(io.BytesIO(yuklenen[ad])) as z:
    z.extractall("veri")

def jsonl_oku(yol):
    return [json.loads(s) for s in open(yol, encoding="utf-8")]

train_ham = jsonl_oku("veri/train.jsonl")
val_ham   = jsonl_oku("veri/val.jsonl")
print(f"train: {len(train_ham)}  val: {len(val_ham)}")
print("örnek girdi (ilk 200 karakter):\n", train_ham[0]["prompt"][:200])

In [ ]:
#@title 3) Sohbet formatına çevir
from datasets import Dataset

def sohbete_cevir(kayitlar):
    return Dataset.from_list([
        {"messages": [
            {"role": "user", "content": k["prompt"]},
            {"role": "assistant", "content": k["cikti"]},
        ]} for k in kayitlar
    ])

train_ds = sohbete_cevir(train_ham)
val_ds   = sohbete_cevir(val_ham)
print(train_ds)

In [ ]:
#@title 4) Taban modeli 4-bit yükle (≈5 dk)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Birincil aday (plan Bölüm 2). Sorun çıkarsa alternatif: "Qwen/Qwen2.5-7B-Instruct"
MODEL_ID = "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("model hazır:", MODEL_ID)

In [ ]:
#@title 5) LoRA + eğitim ayarları (Drive'a checkpoint'li, kopmaya dayanıklı)
from google.colab import drive
drive.mount('/content/drive')

import inspect, torch, trl
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
print("trl sürümü:", trl.__version__)

CIKTI_DIR = "/content/drive/MyDrive/disasterai_lora/qlora_cikti"

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear", task_type="CAUSAL_LM",
)

istenen = dict(
    output_dir=CIKTI_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    fp16=True,
    max_length=1024,
    packing=False,
    report_to="none",
)
gecerli = set(inspect.signature(SFTConfig.__init__).parameters)
kullanilan = istenen if "kwargs" in gecerli else {k: v for k, v in istenen.items() if k in gecerli}
ayar = SFTConfig(**kullanilan)

tr_gecerli = set(inspect.signature(SFTTrainer.__init__).parameters)
ek = {"processing_class": tokenizer} if "processing_class" in tr_gecerli else {"tokenizer": tokenizer}
egitici = SFTTrainer(model=model, args=ayar,
                     train_dataset=train_ds, eval_dataset=val_ds,
                     peft_config=lora, **ek)

# T4 uyumu: eğitilebilir (LoRA) parametreleri fp32'ye sabitle
n = 0
for p in egitici.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float(); n += 1
print(f"{n} parametre fp32'ye sabitlendi — eğitici hazır (checkpoint: Drive)")

In [ ]:
#@title 6) EĞİTİM (kaldığı checkpoint'ten devam eder; ~30-40 dk'da bir sekmeye tıkla)
import glob
ckler = sorted(glob.glob(f"{CIKTI_DIR}/checkpoint-*"),
               key=lambda p: int(p.split("-")[-1]))
if ckler:
    print("checkpoint bulundu, devam ediliyor:", ckler[-1])
    egitici.train(resume_from_checkpoint=ckler[-1])
else:
    print("sıfırdan başlıyor")
    egitici.train()
print("eğitim bitti ✅")

In [ ]:
#@title 7) Adaptörü Drive'a kaydet
from google.colab import drive
drive.mount("/content/drive")

HEDEF = "/content/drive/MyDrive/disasterai_lora"
egitici.save_model(HEDEF)
tokenizer.save_pretrained(HEDEF)
print("kaydedildi:", HEDEF)

In [ ]:
#@title 8) Hızlı deneme — test kümesinden 2 örnek
import json
test_ham = jsonl_oku("veri/test.jsonl")

model.eval()
for k in test_ham[:2]:
    mesajlar = [{"role": "user", "content": k["prompt"]}]
    girdi = tokenizer.apply_chat_template(
        mesajlar, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        cikti = model.generate(girdi, max_new_tokens=350, temperature=0.7,
                               do_sample=True, pad_token_id=tokenizer.eos_token_id)
    uretilen = tokenizer.decode(cikti[0][girdi.shape[1]:], skip_special_tokens=True)
    print("=" * 70)
    print("[TON]", k["ton"], "| [ÖĞRETMENİN YAZDIĞI ilk 150]:")
    print(k["cikti"][:150])
    print("\n[BİZİM MODELİN YAZDIĞI]:")
    print(uretilen[:600])

## Bittikten sonra

- Adaptör `Drive/disasterai_lora` klasöründe. Claude'a "eğitim bitti" de —
  değerlendirme adımını (test kümesinde `dogrula()` + ton doğruluğu + Gemini
  karşılaştırması) Mac'te birlikte koşacağız.
- 8. hücredeki örnek çıktılar ilk göz kontrolü içindir; sayılar CBS verisiyle
  uyumlu görünüyor mu diye bak.